In [1]:
# Load packages
from functools import reduce
import numpy as np 
import pandas as pd 
import os 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261020 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [2]:
import os
import pandas as pd
import gcsfs
import ast

input_path = f'{os.environ["WORKSPACE_BUCKET"]}/data/hail/'
output_path = f'{os.environ["WORKSPACE_BUCKET"]}/data/chip'

fs = gcsfs.GCSFileSystem()

files = fs.ls(input_path)

total_pass = 0
total_fail = 0

dfs = [] 

for f in files:
    if f.endswith('.tsv.bgz'):

        print(f"Processing: {f}")

        file_pass = 0
        file_fail = 0

        with fs.open(f, 'rb') as fh:
            for df in pd.read_csv(
                fh,
                sep='\t',
                compression='gzip',
                chunksize=1000
            ):
                # Keep PASS entries
                df = df[df['FT'] == 'PASS'].copy()

                if df.empty:
                    continue
                    
                # Define variant id     
                df["alleles"] = df["alleles"].apply(ast.literal_eval)

                df["ref"] = df["alleles"].str[0]
                df["alt"] = df["alleles"].str[1]

                df['id'] = df['locus'] + ':' + df['ref'] + ':' + df['alt']
            
                # Count variants failing either threshold
                fail_mask = (
                    (df['DP'] < 20) |
                    ((df['AB'] < 0.35) & (df['AB'] > 0.65)) 
                )
                
                dfs.append(df[fail_mask]) 

                count = fail_mask.sum()

                file_pass += len(df)
                file_fail += count
                
                if not (len(df[fail_mask]) == count):
                    print("Mismatch between fail count") 
                    break 

        total_pass += file_pass
        total_fail += file_fail

        print(f"PASS entries: {file_pass:,}")
        print(f"Failing entries: {file_fail:,}")
        print(
            f"Failing proportion: "
            f"{file_fail / file_pass:.4%}"
            if file_pass else "No PASS entries"
        )

print("\n=== Overall results ===")
print(f"Total PASS entries: {total_pass:,}")
print(f"Total failing entries: {total_fail:,}")
print(
    f"Overall failing proportion: {total_fail / total_pass:.4%}"
    if total_pass else "No PASS entries"
)

# Concatenate dataframes
failed = pd.concat(dfs)

# Reanme columns 
failed.rename(columns={'id':'variant',
                       's':'sample'}, 
              inplace=True)

# Check the total fail 
print(f"Check mistamch failing entries: {(len(failed) == total_fail)}")

# Write locally first
local_file = f"{os.getcwd()}/remove.txt"
gcs_file = f"{output_path}/remove.txt"

failed[['variant', 'sample']].to_csv(
    local_file,
    sep='\t',
    index=False
)

subprocess.run(
    ["gsutil", "cp", local_file, gcs_file],
    check=True
)

print(f"Uploaded: {gcs_file}")

Processing: rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr15_90080000_90110000.tsv.bgz
PASS entries: 1,616,677
Failing entries: 462,620
Failing proportion: 28.6155%
Processing: rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr17_76730000_76740000.tsv.bgz
PASS entries: 3,436,119
Failing entries: 7,175
Failing proportion: 0.2088%
Processing: rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr2_25230000_25320000.tsv.bgz
PASS entries: 1,790,254
Failing entries: 147,274
Failing proportion: 8.2264%
Processing: rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr4_105140000_105280000.tsv.bgz
PASS entries: 4,067,332
Failing entries: 354,253
Failing proportion: 8.7097%
Processing: rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr5_177510000_177520000.tsv.bgz
PASS entries: 1,247,534
Failing entries: 1,044
Failing proportion: 0.0837%

=== Overall results ===
Total PASS entries: 12,157,916
T

Copying file:///home/dataproc/workspace/rw-migration-aou-rw-9f879fc4/notebooks/hail/remove.txt [Content-Type=text/plain]...
\ [1 files][ 27.5 MiB/ 27.5 MiB]                                                
Operation completed over 1 objects/27.5 MiB.                                     


Uploaded: gs://rw-migration-aou-rw-9f879fc4/data/chip/remove.txt
